# Vocal-Guard Inference Notebook

Notebook ini berisi script bersih untuk memuat model dan melakukan prediksi single sample secara offline.

In [1]:
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
import tensorflow as tf
import audioprocessor

In [2]:
TARGET_SAMPLE_RATE = 16000
TARGET_IMAGE_SIZE = (224, 224)
WINDOW_SEC = 1.0

MODEL_LA_PATH = Path('models/voice_detector_LA.h5')
MODEL_PA_PATH = Path('models/voice_detector_PA.h5')
TEST_MANIFEST_PATH = Path('test_data/test_windows.csv')

model_la = tf.keras.models.load_model(MODEL_LA_PATH)
model_pa = tf.keras.models.load_model(MODEL_PA_PATH)
test_df = pd.read_csv(TEST_MANIFEST_PATH)

lfcc_converter = audioprocessor.LfccSpectrogramConverter(
    sample_rate=TARGET_SAMPLE_RATE,
    pre_emphasis_alpha=0.97,
    frame_length_ms=25.0,
    frame_step_ms=10.0,
    num_linear_filters=40,
    num_cepstral_coeffs=20,
    fft_size=512,
)

In [3]:
def audio_to_spectrogram(audio, sample_rate, target_size=TARGET_IMAGE_SIZE):
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    if len(audio) == 0:
        audio = np.zeros(sample_rate, dtype=np.float32)

    if sample_rate != TARGET_SAMPLE_RATE:
        raise ValueError(f'Expected sample_rate={TARGET_SAMPLE_RATE}, got {sample_rate}.')

    rust_out = lfcc_converter.convert_window(audio.astype(np.float32).tolist())
    spec = np.array(rust_out, dtype=np.float32).reshape(3, target_size[0], target_size[1])
    spec = np.transpose(spec, (1, 2, 0))
    spec = spec - spec.min()
    spec = spec / (spec.max() - spec.min() + 1e-8)
    return spec.astype(np.float32)

def predict_row(row):
    audio_path = Path(row['audio_path'])
    start_sample = int(row['window_start_sample'])
    window_frames = int(row['window_frames'])
    sample_rate = int(row['sample_rate'])

    audio, sr = sf.read(str(audio_path), start=start_sample, frames=window_frames, dtype='float32', always_2d=False)
    sample_rate = sr

    target_length = int(sample_rate * WINDOW_SEC)
    if len(audio) < target_length:
        pad_len = target_length - len(audio)
        if audio.ndim == 1:
            audio = np.pad(audio, (0, pad_len))
        else:
            audio = np.pad(audio, ((0, pad_len), (0, 0)))

    spec = audio_to_spectrogram(audio, sample_rate)
    x = np.expand_dims(spec, axis=0)

    p_la = float(model_la.predict(x, verbose=0)[0][0])
    p_pa = float(model_pa.predict(x, verbose=0)[0][0])
    p_ens = (p_la + p_pa) / 2.0
    pred_label = int(p_ens >= 0.5)

    return {
        'window_id': row['window_id'],
        'true_label': int(row['label']),
        'p_la': p_la,
        'p_pa': p_pa,
        'p_ensemble': p_ens,
        'pred_label': pred_label
    }

In [4]:
sample_row = test_df.iloc[0]
result = predict_row(sample_row)
print('Inference result:')
for k, v in result.items():
    print(f'  {k}: {v}')

Inference result:
  window_id: LA_LA_E_9806004_w0005
  true_label: 0
  p_la: 0.5094430446624756
  p_pa: 0.4053037166595459
  p_ensemble: 0.45737338066101074
  pred_label: 0
